**1. REad files using DataStreamReader API**

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType

defined_schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("customer_name", StringType()),
    StructField("date_of_birth", DateType()),
    StructField("telephone", StringType()),
    StructField("email", StringType()),
    StructField("member_since", DateType()),
    StructField("created_timestamp", TimestampType())
])

In [0]:
df = (
    spark.readStream
    .format("JSON")
    .schema(defined_schema)
    .load("/Volumes/gizmobox/landing/operational_data/customers_stream/")
)

**2. Transform the dataframe to add the following columns**
- file_path : Clound file path
- ingestion_date :  Current timestamp

In [0]:
from pyspark.sql.functions import current_timestamp, col
df_transformed_data = (
    df
    .withColumn("file_path", col("_metadata.file_path"))
    .withColumn("ingestion_date", current_timestamp())
)



**3. Write the transformed data stream to delta table**

In [0]:
streaming_query = (
df_transformed_data.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_stream/_checkpoint_stream")
    .toTable("gizmobox.bronze.customer_stream")
)

In [0]:
df = spark.table("gizmobox.bronze.customer_stream")
display(df)

In [0]:
streaming_query.stop()